In [1]:
import pandas as pd
import numpy as np
import json
import ast
from IPython.display import display, Markdown

In [2]:
# Nettoyage du dataframe et Normalisation de la colonne Hotels pour préparer le dataset final pour l'EDA. 

In [3]:
final_df = pd.read_csv("/Users/nicolasbour/Desktop/Projets_IA /Projets_certif/02_kayak/data/tests/final_data.csv")

In [4]:
final_df.head(2)

,city,place_id,lat,lon,base,visibility,dt,timezone,id,name,...,sys.sunrise,sys.sunset,sys.type,sys.id,rain.1h,weather.0.id,weather.0.main,weather.0.description,weather.0.icon,hotels
0,mont saint michel,281361808,48.635954,-1.511460,stations,10000,1788426584,7200,6435453,Huisnes-sur-Mer,...,1788413148,1788461136,NaN,NaN,0,804,Clouds,couvert,04d,"[{'hotel_city': 'Beauvoir', 'name': 'La Villa ..."
1,saint malo,285565834,48.649518,-2.026041,stations,10000,1788426585,7200,2978640,Saint-Malo,...,1788413270,1788461260,1.0,6557.0,0,804,Clouds,couvert,04d,"[{'hotel_city': 'Saint-Malo', 'name': 'Kyriad ..."


In [5]:
# Reconvertir la string en vraie liste de dictionnaires (convertion des strings en objets Python)
final_df["hotels"] = final_df["hotels"].apply(ast.literal_eval)

In [6]:
final_df.head(2)

,city,place_id,lat,lon,base,visibility,dt,timezone,id,name,...,sys.sunrise,sys.sunset,sys.type,sys.id,rain.1h,weather.0.id,weather.0.main,weather.0.description,weather.0.icon,hotels
0,mont saint michel,281361808,48.635954,-1.511460,stations,10000,1788426584,7200,6435453,Huisnes-sur-Mer,...,1788413148,1788461136,NaN,NaN,0,804,Clouds,couvert,04d,"[{'hotel_city': 'Beauvoir', 'name': 'La Villa ..."
1,saint malo,285565834,48.649518,-2.026041,stations,10000,1788426585,7200,2978640,Saint-Malo,...,1788413270,1788461260,1.0,6557.0,0,804,Clouds,couvert,04d,"[{'hotel_city': 'Saint-Malo', 'name': 'Kyriad ..."


In [7]:
# Une ligne par hôtel
final_df = final_df.explode("hotels", ignore_index=True)

In [8]:
final_df.head(2)

,city,place_id,lat,lon,base,visibility,dt,timezone,id,name,...,sys.sunrise,sys.sunset,sys.type,sys.id,rain.1h,weather.0.id,weather.0.main,weather.0.description,weather.0.icon,hotels
0,mont saint michel,281361808,48.635954,-1.51146,stations,10000,1788426584,7200,6435453,Huisnes-sur-Mer,...,1788413148,1788461136,NaN,NaN,0,804,Clouds,couvert,04d,"{'hotel_city': 'Beauvoir', 'name': 'La Villa d..."
1,mont saint michel,281361808,48.635954,-1.51146,stations,10000,1788426584,7200,6435453,Huisnes-sur-Mer,...,1788413148,1788461136,NaN,NaN,0,804,Clouds,couvert,04d,"{'hotel_city': 'Huisnes-sur-Mer', 'name': 'Esc..."


In [9]:
# Transformer chaque dictionnaire hôtel en colonnes
hotel_details = pd.json_normalize(final_df["hotels"])

In [10]:
hotel_details.head(2)

,hotel_city,name,price,score,latitude,longitude
0,Beauvoir,La Villa du Manoir,€ 521,"Avec une note de 9,8 9,8 Exceptionnel 6 expéri...",48.596052,-1.500324
1,Huisnes-sur-Mer,Escale du Mont,€ 364,"Avec une note de 7,5 7,5 Bien 2 expériences vé...",48.621474,-1.447158


In [11]:
hotel_details['name'] = hotel_details['name'].str.split(' - ').str[0] # register hotel label
hotel_details['rate'] = hotel_details['score'].str.extract(r'(?:de\s+)?(\d+(?:,\d+)?)') # extract hotel score  
hotel_details['rate_label'] = hotel_details['score'].str.extract(r'(?:\d+(?:,\d+)?)\s+(?:\d+(?:,\d+)?)\s+([^\d]+?)\s+\d+')[0].str.strip() # extract rate label
hotel_details['experience'] = hotel_details['score'].str.extract(r'(\d+(?:\s+\d+)*)\s+expériences?')[0] # extract number of experiences
hotel_details['hotel_city'] = hotel_details['hotel_city'].str.split(',').str[-1].str.strip() # extract hotel city

In [12]:
hotel_details.head(2)

,hotel_city,name,price,score,latitude,longitude,rate,rate_label,experience
0,Beauvoir,La Villa du Manoir,€ 521,"Avec une note de 9,8 9,8 Exceptionnel 6 expéri...",48.596052,-1.500324,"9,8",Exceptionnel,6
1,Huisnes-sur-Mer,Escale du Mont,€ 364,"Avec une note de 7,5 7,5 Bien 2 expériences vé...",48.621474,-1.447158,"7,5",Bien,2


In [13]:
hotel_details = pd.concat([final_df.drop(columns=["hotels"]), hotel_details], axis=1)

In [14]:
hotel_details.head(2)

,city,place_id,lat,lon,base,visibility,dt,timezone,id,name,...,weather.0.icon,hotel_city,name,price,score,latitude,longitude,rate,rate_label,experience
0,mont saint michel,281361808,48.635954,-1.51146,stations,10000,1788426584,7200,6435453,Huisnes-sur-Mer,...,04d,Beauvoir,La Villa du Manoir,€ 521,"Avec une note de 9,8 9,8 Exceptionnel 6 expéri...",48.596052,-1.500324,"9,8",Exceptionnel,6
1,mont saint michel,281361808,48.635954,-1.51146,stations,10000,1788426584,7200,6435453,Huisnes-sur-Mer,...,04d,Huisnes-sur-Mer,Escale du Mont,€ 364,"Avec une note de 7,5 7,5 Bien 2 expériences vé...",48.621474,-1.447158,"7,5",Bien,2


!!! Attention !!! il y a un doublon avec le label name : 
    - openweathermap : name => nom de la ville de la station
        "name" =>> "station_name"
    - booking : name => nom de l'hôtel
        "name" =>> "hotel_name"

sys.type = 4 # => info interne à openweathermap obsolète, on peut la supprimer
sys.id = 4 # => info interne à openweathermap obsolète, on peut la supprimer

In [15]:
# Openweathermap name : 
columns = list(hotel_details.columns)
columns[9] = "station_name"
hotel_details.columns = columns

In [16]:
# Hotel name : 
hotel_details.rename(columns={"name": "hotel_name"}, inplace=True)

In [17]:
hotel_details.head(2)

,city,place_id,lat,lon,base,visibility,dt,timezone,id,station_name,...,weather.0.icon,hotel_city,hotel_name,price,score,latitude,longitude,rate,rate_label,experience
0,mont saint michel,281361808,48.635954,-1.51146,stations,10000,1788426584,7200,6435453,Huisnes-sur-Mer,...,04d,Beauvoir,La Villa du Manoir,€ 521,"Avec une note de 9,8 9,8 Exceptionnel 6 expéri...",48.596052,-1.500324,"9,8",Exceptionnel,6
1,mont saint michel,281361808,48.635954,-1.51146,stations,10000,1788426584,7200,6435453,Huisnes-sur-Mer,...,04d,Huisnes-sur-Mer,Escale du Mont,€ 364,"Avec une note de 7,5 7,5 Bien 2 expériences vé...",48.621474,-1.447158,"7,5",Bien,2


In [18]:
hotel_details = hotel_details.drop(columns=["score", "sys.type", "sys.id", "sys.country", "coord.lon", "coord.lat", "cod", "base", "station_name", "id", "timezone", "place_id"])

In [19]:
pd.set_option('display.max_columns', None)  # Afficher toutes les colonnes
hotel_details.head()

,city,lat,lon,visibility,dt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,main.humidity,main.sea_level,main.grnd_level,wind.speed,wind.deg,wind.gust,clouds.all,sys.sunrise,sys.sunset,rain.1h,weather.0.id,weather.0.main,weather.0.description,weather.0.icon,hotel_city,hotel_name,price,latitude,longitude,rate,rate_label,experience
0,mont saint michel,48.635954,-1.51146,10000,1788426584,20.24,20.49,20.24,20.24,1022,83,1022,1017,4.6,217,6.5,99,1788413148,1788461136,0,804,Clouds,couvert,04d,Beauvoir,La Villa du Manoir,€ 521,48.596052,-1.500324,"9,8",Exceptionnel,6
1,mont saint michel,48.635954,-1.51146,10000,1788426584,20.24,20.49,20.24,20.24,1022,83,1022,1017,4.6,217,6.5,99,1788413148,1788461136,0,804,Clouds,couvert,04d,Huisnes-sur-Mer,Escale du Mont,€ 364,48.621474,-1.447158,"7,5",Bien,2
2,mont saint michel,48.635954,-1.51146,10000,1788426584,20.24,20.49,20.24,20.24,1022,83,1022,1017,4.6,217,6.5,99,1788413148,1788461136,0,804,Clouds,couvert,04d,Huisnes-sur-Mer,Horizon du Mont 15 personnes,€ 1 140,48.619514,-1.459112,"9,7",Exceptionnel,10
3,mont saint michel,48.635954,-1.51146,10000,1788426584,20.24,20.49,20.24,20.24,1022,83,1022,1017,4.6,217,6.5,99,1788413148,1788461136,0,804,Clouds,couvert,04d,Beauvoir,Ermitage,€ 1 954,48.594490,-1.511429,"8,7",Superbe,152
4,mont saint michel,48.635954,-1.51146,10000,1788426584,20.24,20.49,20.24,20.24,1022,83,1022,1017,4.6,217,6.5,99,1788413148,1788461136,0,804,Clouds,couvert,04d,Beauvoir,Le Petit Gîte du Mont,€ 440,48.597734,-1.511773,"8,9",Superbe,55


In [20]:
hotel_details.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 875 entries, 0 to 874
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   city                   875 non-null    object 
 1   lat                    875 non-null    float64
 2   lon                    875 non-null    float64
 3   visibility             875 non-null    int64  
 4   dt                     875 non-null    int64  
 5   main.temp              875 non-null    float64
 6   main.feels_like        875 non-null    float64
 7   main.temp_min          875 non-null    float64
 8   main.temp_max          875 non-null    float64
 9   main.pressure          875 non-null    int64  
 10  main.humidity          875 non-null    int64  
 11  main.sea_level         875 non-null    int64  
 12  main.grnd_level        875 non-null    int64  
 13  wind.speed             875 non-null    float64
 14  wind.deg               875 non-null    int64  
 15  wind.g

In [21]:
hotel_details.isna().sum()

city                       0
lat                        0
lon                        0
visibility                 0
dt                         0
main.temp                  0
main.feels_like            0
main.temp_min              0
main.temp_max              0
main.pressure              0
main.humidity              0
main.sea_level             0
main.grnd_level            0
wind.speed                 0
wind.deg                   0
wind.gust                475
clouds.all                 0
sys.sunrise                0
sys.sunset                 0
rain.1h                    0
weather.0.id               0
weather.0.main             0
weather.0.description      0
weather.0.icon             0
hotel_city                 0
hotel_name                 0
price                      0
latitude                   0
longitude                  0
rate                      10
rate_label                10
experience                10
dtype: int64

In [22]:
print(hotel_details[hotel_details["rate"].isna()])

                city        lat       lon  visibility          dt  main.temp  \
81          le havre  49.493898  0.107973       10000  1788426587      19.61   
159           amiens  49.894171  2.295695       10000  1788426591      20.67   
294        eguisheim  48.044797  7.307962       10000  1788426596      23.92   
326            dijon  47.321581  5.041470       10000  1788426302      23.80   
526  aix en provence  43.529842  5.447474       10000  1788426607      25.52   
619            nimes  43.837425  4.360069       10000  1788426303      28.22   
700      carcassonne  43.213036  2.349107       10000  1788426615      25.15   
702      carcassonne  43.213036  2.349107       10000  1788426615      25.15   
755         toulouse  43.604464  1.444243       10000  1788426576      23.97   
822         biarritz  43.471144 -1.552728       10000  1788426619      22.00   

     main.feels_like  main.temp_min  main.temp_max  main.pressure  \
81             20.03          19.51          21.68

In [23]:
percentage_null = (hotel_details["rate"].isna().sum() / len(hotel_details))*100

In [24]:
display(Markdown(f"""
**Hypothèse à valider :** Les valeurs nulles semblent correspondre à des hôtels 
qui n'ont pas de note ou d'avis laissés sur le site Booking. 
Cela représente {percentage_null:.2f}% des hôtels de la base de données. 
Il est donc possible de supprimer ces lignes sans perte d'information significative."""))


**Hypothèse à valider :** Les valeurs nulles semblent correspondre à des hôtels 
qui n'ont pas de note ou d'avis laissés sur le site Booking. 
Cela représente 1.14% des hôtels de la base de données. 
Il est donc possible de supprimer ces lignes sans perte d'information significative.

In [25]:
hotel_details.to_csv("/Users/nicolasbour/Desktop/Projets_IA /Projets_certif/02_kayak/data/final_data_cleaned.csv", index=False)